In [74]:
"Importing commonly used libraries"
import sqlite3 as sql
import pandas as pd 
import numpy as np
import matplotlib.pyplot as mat
import seaborn as sns
import duckdb

In [ ]:
"Loading tables into pandas dataframes"
Customers = pd.read_csv('customers.csv')
Orders = pd.read_csv('orders.csv')
Products = pd.read_csv('products.csv')
Order_Items = pd.read_csv('order_items.csv')
Order_Payments = pd.read_csv('order_payments.csv')
Order_Reviews = pd.read_csv('order_reviews.csv')

In [ ]:
"Droping unnecessary columns"
Orders = Orders[['order_id','customer_id','order_purchase_timestamp']]
Products = Products[['product_id','product_category_name']]
Order_Items = Order_Items[['order_id','price','product_id','freight_value']]
Order_Payments = Order_Payments[['order_id','payment_value']]
Order_Reviews = Order_Reviews[['review_id','order_id','review_score']]

In [28]:
"EDA on unique customers and orders"
no_of_orders = len(Orders)
no_of_unique_orders = Orders["order_id"].nunique()

no_of_customers = len(Customers)
no_of_unique_customers = Customers["customer_unique_id"].nunique()

print(no_of_orders,no_of_unique_orders,no_of_customers,no_of_unique_customers)

99441 99441 99441 96096


In [88]:
"Revenue Analysis"
q1 = """
SELECT 
c.customer_unique_id AS Customer_ID,
count(DISTINCT o.order_id) AS Orders_Amount,
sum(op.payment_value) AS Order_Cost,
DATEDIFF('day',CAST(MAX(o.order_purchase_timestamp) AS DATE),CAST('2018-10-17' AS DATE)) AS Recent_Order
FROM Customers AS c
LEFT JOIN Orders AS o
ON c.customer_id = o.customer_id
LEFT JOIN Order_Payments AS op
on o.order_id = op.order_id
GROUP BY c.customer_unique_id
"""

result_q8 = duckdb.query(q1).to_df()
result_q8


,Customer_ID,Orders_Amount,Order_Cost,Recent_Order
0,299905e3934e9e181bfb2e164dd4b4f8,1,169.76,445
1,ac307db9d15fc5bb19b61298bd6bd1ed,1,423.67,96
2,76c9a12722537319e44c31e70b7815c3,2,99.58,136
3,977136c10acb01bd9cecb7d18ff7d1a0,1,45.95,612
4,c59ba65efe3622bcfaf929ecd5fbfbf5,1,320.18,131
...,...,...,...,...
96091,43fb4e33ebe4ac765e99c7b57e5d6940,1,113.34,295
96092,b0428b000a682156b8856ede42ffa1f1,1,154.81,132
96093,2fbed526955cf9cfa88dfc3a1c73463e,1,408.97,138
96094,d0cca06294eeb5485945d73603e8b949,1,536.30,104


In [120]:
rfm = result_q8.copy()

def rfm_score(series, reverse = False):
    series_no_null = series.dropna()

    bins = pd.qcut(series_no_null, 4, duplicates = "drop")
    n_bins = bins.cat.categories.size
    
    if reverse:
        labels = list(range(n_bins,0,-1))
    else:
        labels = list(range(1,n_bins + 1))
    
    scored = pd.qcut(series_no_null,n_bins,labels)

    return scored.reindex(series.index)

rfm["Recency"] = rfm_score(rfm["Recent_Order"],reverse = True)
rfm["Frequency"] = rfm_score(rfm["Orders_Amount"])
rfm["Monetary"] = rfm_score(rfm["Order_Cost"])

rfm[["Recency","Frequency","Monetary"]] = (
    rfm[["Recency","Frequency","Monetary"]]
    .fillna(1)
    .astype(int)
)

rfm["RFM_Score"] = rfm["Recency"] + rfm["Frequency"] + rfm["Monetary"]

rfm

,Customer_ID,Orders_Amount,Order_Cost,Recent_Order,Recency,Frequency,Monetary,RFM_Score
0,299905e3934e9e181bfb2e164dd4b4f8,1,169.76,445,1,1,3,5
1,ac307db9d15fc5bb19b61298bd6bd1ed,1,423.67,96,4,1,4,9
2,76c9a12722537319e44c31e70b7815c3,2,99.58,136,4,1,2,7
3,977136c10acb01bd9cecb7d18ff7d1a0,1,45.95,612,1,1,1,3
4,c59ba65efe3622bcfaf929ecd5fbfbf5,1,320.18,131,4,1,4,9
...,...,...,...,...,...,...,...,...
96091,43fb4e33ebe4ac765e99c7b57e5d6940,1,113.34,295,2,1,3,6
96092,b0428b000a682156b8856ede42ffa1f1,1,154.81,132,4,1,3,8
96093,2fbed526955cf9cfa88dfc3a1c73463e,1,408.97,138,4,1,4,9
96094,d0cca06294eeb5485945d73603e8b949,1,536.30,104,4,1,4,9
